### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- Hugging_Face: For embeddings and language model (you can substitute with other providers)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Hp\AppData\Local\Temp\ipykernel_22984\3797715575.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [4]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [5]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\Hp\AppData\Local\Temp\tmpp98tftak


In [6]:
temp_dir

'C:\\Users\\Hp\\AppData\\Local\\Temp\\tmpp98tftak'

### 2. PDF Loading

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],

        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Laod PDF

        loader=PyPDFLoader(pdf_path)
        pages=loader.load()

        ## Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

preprocessor=SmartPDFProcessor()
## Process a PDF if available
try:
    smart_chunks=preprocessor.process_pdf("data/Quantum_Computing.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")
    
            


Processed into 102 smart chunks

Sample chunk metadata:
  producer: Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)
  creator: PyPDF
  creationdate: 2022-06-10T20:33:56+05:30
  moddate: 2022-06-13T16:26:49-04:00
  ieee article id: 9783210
  ieee issue id: 9680797
  subject: IEEE Open Journal of Nanotechnology;2022;3; ;10.1109/OJNANO.2022.3178545
  ieee publication id: 8782713
  title: Quantum Computing: Fundamentals, Implementations and Applications
  source: data/Quantum_Computing.pdf
  total_pages: 17
  page: 1
  page_label: 1
  chunk_method: smart_pdf_processor
  char_count: 4310


In [8]:
smart_chunks

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2022-06-10T20:33:56+05:30', 'moddate': '2022-06-13T16:26:49-04:00', 'ieee article id': '9783210', 'ieee issue id': '9680797', 'subject': 'IEEE Open Journal of Nanotechnology;2022;3; ;10.1109/OJNANO.2022.3178545', 'ieee publication id': '8782713', 'title': 'Quantum Computing: Fundamentals, Implementations and Applications', 'source': 'data/Quantum_Computing.pdf', 'total_pages': 17, 'page': 1, 'page_label': '1', 'chunk_method': 'smart_pdf_processor', 'char_count': 4310}, page_content='Received 28 January 2022; revised 23 April 2022; accepted 21 May 2022. Date of publication 27 May 2022; date of current version 13 June 2022. The review of this article was arranged by Associate Editor Kazuhiko Endo. Digital Object Identifier 10.1109/OJNANO.2022.3178545 Quantum Computing: Fundamentals, Implementations and Applications 

### Embedding Models

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

sample_text = "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10571.72it/s]


In [10]:
vector=embeddings.embed_query(sample_text)
vector

[0.008409216068685055,
 -0.003605820005759597,
 0.05522147938609123,
 0.06061019375920296,
 0.031035901978611946,
 -0.024385986849665642,
 -0.029740167781710625,
 -0.030584225431084633,
 -0.07072202861309052,
 -0.002816963940858841,
 -0.0427212119102478,
 0.011185092851519585,
 0.05198772996664047,
 -0.09259217977523804,
 -0.01497737318277359,
 0.03200266510248184,
 0.0031231248285621405,
 0.0004823635972570628,
 -0.058316901326179504,
 -0.06758233904838562,
 0.017395272850990295,
 -0.014859731309115887,
 -0.04645839333534241,
 0.028660951182246208,
 -0.010227466002106667,
 0.052250005304813385,
 0.03158418834209442,
 0.06607231497764587,
 0.018648430705070496,
 -0.016719480976462364,
 0.03997329995036125,
 -0.01592491939663887,
 0.03211919218301773,
 -0.004316603299230337,
 -0.028114663437008858,
 0.04597124457359314,
 -0.03944206237792969,
 0.07581846415996552,
 0.04164663702249527,
 -0.03430870175361633,
 -0.054628968238830566,
 -0.03818632289767265,
 0.01733567751944065,
 -0.018334

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [11]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Hugging Face embeddings
vectorstore=Chroma.from_documents(
    documents=smart_chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 918 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [12]:
query="TOP 12 MOST PROLIFIC COUNTRIES WORLDWIDE IN QUANTUM COMPUTING RESEARCH"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/Quantum_Computing.pdf', 'ieee article id': '9783210', 'chunk_method': 'smart_pdf_processor', 'subject': 'IEEE Open Journal of Nanotechnology;2022;3; ;10.1109/OJNANO.2022.3178545', 'page': 2, 'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'page_label': '2', 'ieee publication id': '8782713', 'total_pages': 17, 'ieee issue id': '9680797', 'moddate': '2022-06-13T16:26:49-04:00', 'creationdate': '2022-06-10T20:33:56+05:30', 'title': 'Quantum Computing: Fundamentals, Implementations and Applications', 'char_count': 3769, 'creator': 'PyPDF'}, page_content='showed up as jour- nals, 22.88% (4525) as conference papers, 6.20% (1228) as book series, 2.40% (480) as books, and 0.8% (154) as trade journals and others as shown in Fig. 2. FIGURE 1. Publications in quantum computing research across the World. FIGURE 2. Source-type wise Quantum computing research in world. C. TOP 12 MOST PROLIFIC CO

In [13]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: TOP 12 MOST PROLIFIC COUNTRIES WORLDWIDE IN QUANTUM COMPUTING RESEARCH

Top 3 similar chunks:

--- Chunk 1 ---
showed up as jour- nals, 22.88% (4525) as conference papers, 6.20% (1228) as book series, 2.40% (480) as books, and 0.8% (154) as trade journals and others as shown in Fig. 2. FIGURE 1. Publications i...
Source: data/Quantum_Computing.pdf

--- Chunk 2 ---
showed up as jour- nals, 22.88% (4525) as conference papers, 6.20% (1228) as book series, 2.40% (480) as books, and 0.8% (154) as trade journals and others as shown in Fig. 2. FIGURE 1. Publications i...
Source: data/Quantum_Computing.pdf

--- Chunk 3 ---
showed up as jour- nals, 22.88% (4525) as conference papers, 6.20% (1228) as book series, 2.40% (480) as books, and 0.8% (154) as trade journals and others as shown in Fig. 2. FIGURE 1. Publications i...
Source: data/Quantum_Computing.pdf


### Advanced Similarity Search With Scores

In [14]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'chunk_method': 'smart_pdf_processor', 'page_label': '2', 'ieee publication id': '8782713', 'creationdate': '2022-06-10T20:33:56+05:30', 'title': 'Quantum Computing: Fundamentals, Implementations and Applications', 'total_pages': 17, 'page': 2, 'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'char_count': 3769, 'subject': 'IEEE Open Journal of Nanotechnology;2022;3; ;10.1109/OJNANO.2022.3178545', 'ieee article id': '9783210', 'ieee issue id': '9680797', 'source': 'data/Quantum_Computing.pdf', 'moddate': '2022-06-13T16:26:49-04:00'}, page_content='showed up as jour- nals, 22.88% (4525) as conference papers, 6.20% (1228) as book series, 2.40% (480) as books, and 0.8% (154) as trade journals and others as shown in Fig. 2. FIGURE 1. Publications in quantum computing research across the World. FIGURE 2. Source-type wise Quantum computing research in world. C. TOP 12 MOST PROLIFIC C

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [15]:


from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

llm.invoke("What are the top 12 most prolific countries worldwide in quantum computing research?")


AIMessage(content=[{'type': 'text', 'text': 'Determining the most prolific countries in quantum computing (QC) research involves looking at a combination of metrics: **academic publication volume, citation impact, patent filings, government funding, and private sector activity** (such as startups and tech giants). \n\nWhile the **United States** and **China** are the clear global leaders, several other nations have established world-class research ecosystems. Based on data from the McKinsey Quantum Technology Monitor, the World Intellectual Property Organization (WIPO), and academic databases (like Web of Science and arXiv), here are the top 12 most prolific countries in quantum computing research.\n\n---\n\n### 1. United States\n*   **Strengths:** Private-sector dominance, venture capital, and high-impact academic research.\n*   **Key Institutions/Companies:** IBM, Google, Honeywell (Quantinuum), Rigetti, MIT, Harvard, Caltech, Chicago Quantum Exchange.\n*   **Overview:** The U.S. is 

### Modern RAG Chain

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [17]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E70F2B0440>, search_kwargs={})

In [18]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [19]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [20]:
### Create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [21]:
### Create The Final RAG Chain
from langchain_classic.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E70F2B0440>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you

In [22]:
response=rag_chain.invoke({"input":"Who are contributors to quantum computing research?"})

In [23]:
response

{'input': 'Who are contributors to quantum computing research?',
 'context': [Document(metadata={'creationdate': '2022-06-10T20:33:56+05:30', 'ieee publication id': '8782713', 'source': 'data/Quantum_Computing.pdf', 'ieee issue id': '9680797', 'subject': 'IEEE Open Journal of Nanotechnology;2022;3; ;10.1109/OJNANO.2022.3178545', 'total_pages': 17, 'title': 'Quantum Computing: Fundamentals, Implementations and Applications', 'moddate': '2022-06-13T16:26:49-04:00', 'creator': 'PyPDF', 'page_label': '1', 'ieee article id': '9783210', 'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'char_count': 4310, 'chunk_method': 'smart_pdf_processor', 'page': 1}, page_content='may have merit. Quantum computing is attracting more and more inter- est of industrial sectors, not only broad-interest corporations like Microsoft or Google, but also companies more tradition- ally linked to the area of nanoelectronics and nanotechnology (e.g

In [24]:
response['answer']

'Contributors to quantum computing research include major corporations such as Microsoft and Google, as well as nanotechnology-focused companies like IBM and Intel. Additionally, D-Wave Systems, a Canadian organization, is a contributor that developed the first quantum computer out of superconductors in 1999.'

In [25]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})
    
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: I do not know the answer to this question because the provided context does not contain information about the types of machine learning.

Retrieved Context:

--- Source 1 ---
BHAT ET AL.: QUANTUM COMPUTING: FUNDAMENTALS, IMPLEMENTATIONS AND APPLICATIONS gates exhibit the property of reversibility. Hence using quan- tum computer, we can overcome the irreversibility nature o...

--- Source 2 ---
BHAT ET AL.: QUANTUM COMPUTING: FUNDAMENTALS, IMPLEMENTATIONS AND APPLICATIONS gates exhibit the property of reversibility. Hence using quan- tum computer, we can overcome the irreversibility nature o...

--- Source 3 ---
BHAT ET AL.: QUANTUM COMPUTING: FUNDAMENTALS, IMPLEMENTATIONS AND APPLICATIONS gates exhibit the property of reversibility. Hence using quan- tum computer, we can overcome the irreversibility nature o...

--- Source 4 ---
BHAT ET AL.: QUANTUM COMPUTING: FUNDAMENTALS

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [26]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [27]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [28]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E70F2B0440>, search_kwargs={})

In [29]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E70F2B0440>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens'

In [31]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

"Based on the provided context, I don't know the answer to what Deep Learning is, as the term is not mentioned in the text."

In [ ]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [38]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What is quantum computing?")

Testing LCEL Chain:
Question: What is quantum computing?
--------------------------------------------------
Answer: Based on the provided context, quantum computing can be understood through the following details:

* **Definition and Field:** The study of quantum computing is a "subfield of quantum information science." Along with quantum information, quantum computation is defined as "the study of the information processing tasks that can be accomplished using quantum mechanical systems."
* **How it Works:** Unlike current computers that manipulate individual bits, quantum computers "make use of a quantum-mechanical phenomenon (e.g., superposition and entanglement) that allows data to be represented as quantum bits (qubits)." These qubits are "not constrained to conventional 0."


AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'